# HD art pilot: backdrops for The CurioKids Club and AInation Studio

Makes **10 background pictures**, each in **portrait (1080×1920, Shorts)** and **landscape (1920×1080, long videos)**, with **2 versions** of each (40 PNGs), then zips them as `art_pilot.zip`.

* Model: **FLUX.1-schnell** by Black Forest Labs, **Apache 2.0** licence (free for commercial use, including YouTube monetisation).
* Free Kaggle GPU. No API keys, no paid services. Nothing is posted anywhere.

**Before you run** (right-hand panel → *Session options*):
1. **Accelerator → GPU T4 x2** (or GPU P100).
2. **Internet → On**.

Both need a phone-verified Kaggle account (Settings → Phone verification).

**One-time Hugging Face step** (the model is free, but its download needs a login):
1. Sign up free at huggingface.co.
2. Open huggingface.co/black-forest-labs/FLUX.1-schnell and click **Agree and access repository** (approved instantly).
3. Go to huggingface.co → Settings → **Access Tokens** → **Create new token** → type **Read** → copy it.
4. In this notebook: menu **Add-ons → Secrets → Add secret**. Name: `HF_TOKEN`, value: the token. Tick the box so this notebook can use it.

Keep the token only in Kaggle Secrets. Don't paste it into chat or into the notebook.

**Run:** click **Save Version** (top right) → **Save & Run All (Commit)** → **Save**. It runs in the background (about 30–60 min), so you can close the tab. When it finishes, open the version → **Output** → download **art_pilot.zip** and send it to Claude.

In [ ]:
# 1) Install the libraries (about 1-2 minutes)
!pip install -q -U "diffusers>=0.32" "transformers>=4.46" accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# 2) Check the GPU
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 (Session options, right-hand panel) and run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
# T4/P100 have no fast bfloat16, so they use float16; newer GPUs use bfloat16
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("compute type:", DTYPE)

In [ ]:
# 3) The pictures. Edit prompts here if you want; keep "no people, no text".
KIDS_STYLE = ("soft 3D clay-like children's picture-book illustration, rounded friendly shapes, warm pastel colours, "
              "gentle global illumination, soft shadows, shallow depth of field, cheerful and calm, high detail, "
              "no people, no characters, no animals unless named, no text, no letters, no logos, no watermark")
AIN_STYLE = ("clean modern 3D render, soft studio lighting, white and light grey with one electric blue accent (#2F6BFF), "
             "minimal and uncluttered, subtle depth of field, premium tech look, high detail, "
             "no people, no text, no letters, no numbers, no logos, no watermark, no screens with writing")

PICTURES = [
  # (channel, name, what to show)  - the middle and the bottom stay calm: characters stand there and boards go on top
  ("kids", "garden",   "a sunny garden with soft grass, round bushes, a few big flowers at the sides, blue sky with fluffy clouds, open empty lawn in the middle"),
  ("kids", "playroom", "a bright cosy children's playroom, toy shelves and building blocks at the sides, a big round window with daylight, empty soft rug in the middle"),
  ("kids", "space",    "a friendly outer-space scene, deep blue and purple sky full of twinkling stars, a ringed planet and a small moon at the edges, open space in the middle"),
  ("kids", "sprout",   "close-up of a tiny green sprout with two leaves pushing out of dark rich soil, morning sunlight, dew drops, blurred garden behind"),
  ("kids", "waterdrop","a big shiny water droplet about to fall from a green leaf, sparkling reflections, soft rain and a faint rainbow in a blurred background"),
  ("kids", "earth",    "planet Earth seen from space, bright blue oceans, green land and white clouds, glowing atmosphere, soft starry background, Earth at the lower half"),
  ("ain",  "studio",   "an empty bright modern tech studio backdrop, soft curved white wall, thin glowing blue light strips at the edges, clean floor, lots of empty space"),
  ("ain",  "brain",    "a glowing abstract artificial-intelligence brain made of fine blue light lines and small nodes, floating on a soft white background"),
  ("ain",  "desk",     "a tidy modern desk seen from a slight angle, a closed silver laptop, a small plant and a coffee cup, soft window daylight, white and blue tones, blurred background"),
  ("ain",  "network",  "an abstract network of softly glowing blue dots connected by thin lines, gentle depth, white and pale blue background, calm and spacious"),
]
SEEDS = [11, 42]                      # two versions of each picture; the same seed gives the same picture again
SIZES = {"portrait": (864, 1536, (1080, 1920)), "landscape": (1536, 864, (1920, 1080))}   # made at ~1.3 MP, then resized
print(len(PICTURES) * len(SEEDS) * len(SIZES), "pictures to make")

In [ ]:
# 4) Load FLUX.1-schnell in 4-bit so it fits on one 16 GB GPU (first time downloads ~33 GB; about 5-10 minutes)
import os, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb
MODEL = "black-forest-labs/FLUX.1-schnell"
# Needs the HF_TOKEN Kaggle secret (see the steps at the top)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using the HF_TOKEN secret")
except Exception:
    raise SystemExit("No HF_TOKEN secret: do the one-time Hugging Face step at the top (Add-ons -> Secrets -> HF_TOKEN), then run again.")
q = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
transformer = FluxTransformer2DModel.from_pretrained(MODEL, subfolder="transformer", quantization_config=DiffBnb(**q), torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder_2", quantization_config=TfBnb(**q), torch_dtype=DTYPE)
pipe = FluxPipeline.from_pretrained(MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache()
print("model ready")

In [ ]:
# 5) Make the pictures (about 1 minute each on a T4). Already-made files are skipped, so a re-run continues where it stopped.
import json, time, datetime
import numpy as np
from PIL import Image
OUT = "/kaggle/working/art_pilot"
os.makedirs(OUT, exist_ok=True)
manifest = []
t_start = time.time()
for ch, name, what in PICTURES:
    style = KIDS_STYLE if ch == "kids" else AIN_STYLE
    prompt = f"{what}, {style}"
    for orient, (w, h, final) in SIZES.items():
        for seed in SEEDS:
            fn = f"{ch}/{name}_{orient}_s{seed}.png"
            path = os.path.join(OUT, fn)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            if not os.path.exists(path):
                t = time.time()
                img = pipe(prompt, width=w, height=h, num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256,
                           generator=torch.Generator("cpu").manual_seed(seed)).images[0]
                if np.asarray(img).mean() < 3:  # all-black picture = number overflow on this GPU
                    print("WARNING: black picture for", fn, "- tell Claude (it means the float16 setting needs changing)")
                img.resize(final, Image.LANCZOS).save(path, optimize=True)
                print(f"made {fn} in {time.time() - t:.0f}s")
            manifest.append({"file": fn, "channel": ch, "name": name, "orientation": orient, "seed": seed,
                             "size": list(final), "made_at": list((w, h)), "prompt": prompt,
                             "model": MODEL, "steps": 4, "licence": "Apache-2.0 (FLUX.1-schnell)",
                             "date": datetime.date.today().isoformat()})
json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=1)
print(f"done: {len(manifest)} pictures in {(time.time() - t_start) / 60:.0f} min")

In [ ]:
# 6) Contact sheet to look at, and the zip to download (Output -> art_pilot.zip)
from PIL import ImageDraw
thumbs = []
for m in manifest:
    im = Image.open(os.path.join(OUT, m["file"])).convert("RGB")
    im.thumbnail((270, 270)); tile = Image.new("RGB", (280, 300), "white"); tile.paste(im, ((280 - im.width) // 2, 5))
    ImageDraw.Draw(tile).text((6, 282), m["file"].split("/")[-1][:40], fill="black"); thumbs.append(tile)
cols = 8; rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 280, rows * 300), "white")
for i, t in enumerate(thumbs): sheet.paste(t, ((i % cols) * 280, (i // cols) * 300))
sheet.save(os.path.join(OUT, "contact_sheet.jpg"), quality=85)
import shutil
shutil.make_archive("/kaggle/working/art_pilot", "zip", OUT)
print("zip:", round(os.path.getsize("/kaggle/working/art_pilot.zip") / 2**20, 1), "MB")
sheet